# EndoSeg — MMOTU Dataset Exploration

Quick EDA of the MMOTU OTU_2d ovarian-tumor ultrasound dataset.
Run `bash data/download_mmotu.sh` first to fetch the images.

Covers:
- Class distribution (8 ovarian-tumor categories)
- Train/val/test split sizes (from `splits.json` if preprocessing has run)
- Visual grid of 8 random image + mask pairs
- Per-class histogram
- Image size statistics

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Adjust if running from a different working directory
ROOT = Path("../data/MMOTU_DataSet")
PROCESSED = Path("../data/processed")
IMAGES_DIR = ROOT / "images"
MASKS_DIR  = ROOT / "annotations"
LAYERS_DIR = ROOT / "8_layers"

assert ROOT.exists(), f"Dataset not found at {ROOT}. Run: bash data/download_mmotu.sh"
print(f"Dataset root : {ROOT.resolve()}")
print(f"Images       : {len(list(IMAGES_DIR.glob('*.JPG')))}")
print(f"Masks        : {len(list(MASKS_DIR.glob('*.PNG')))}")

In [ ]:
# --- Class distribution from 8_layers/ sub-folders ---
class_counts = {}
for d in sorted(LAYERS_DIR.iterdir()):
    if d.is_dir():
        count = len(list(d.glob("*.JPG")))
        class_counts[d.name] = count

print("8-class distribution (8_layers/):")
for name, count in class_counts.items():
    bar = "#" * (count // 10)
    print(f"  {name:<38} {count:>4}  {bar}")
print(f"  {'TOTAL':<38} {sum(class_counts.values()):>4}")

# --- Train/val/test sizes (from splits.json if preprocessing has been run) ---
splits_path = PROCESSED / "splits.json"
if splits_path.exists():
    with open(splits_path) as f:
        splits = json.load(f)
    print("\nPatient-disjoint splits (splits.json):")
    for split, items in splits.items():
        print(f"  {split:<6}: {len(items)} samples")
else:
    print(f"\nsplits.json not found at {splits_path}")
    print("Run: python jobs/preprocess/preprocess.py --raw-dir data/MMOTU_DataSet --out-dir data/processed")

In [ ]:
# --- Visualise 8 random (image, mask) pairs in a 2-row grid ---
random.seed(42)
all_images = sorted(IMAGES_DIR.glob("*.JPG"))
samples = random.sample(all_images, min(8, len(all_images)))

fig, axes = plt.subplots(2, len(samples), figsize=(3 * len(samples), 6))

for col, img_path in enumerate(samples):
    mask_path = MASKS_DIR / (img_path.stem + ".PNG")

    img = np.asarray(Image.open(img_path).convert("RGB"))
    axes[0, col].imshow(img)
    axes[0, col].set_title(img_path.stem, fontsize=7)
    axes[0, col].axis("off")

    if mask_path.exists():
        mask = np.asarray(Image.open(mask_path))
        if mask.ndim == 3:
            mask = mask[..., 0]
        axes[1, col].imshow(mask, cmap="viridis")
        axes[1, col].set_title(f"vals:{np.unique(mask).tolist()}", fontsize=6)
    else:
        axes[1, col].text(0.5, 0.5, "no mask", ha="center", va="center",
                          transform=axes[1, col].transAxes)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("image", fontsize=11)
axes[1, 0].set_ylabel("mask", fontsize=11)
fig.suptitle("MMOTU — 8 random samples", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Per-class histogram ---
names  = list(class_counts.keys())
counts = list(class_counts.values())
short  = [n.replace("_", "\n") for n in names]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(names)), counts, color="steelblue", edgecolor="white")
ax.bar_label(bars, fontsize=9)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(short, fontsize=9)
ax.set_ylabel("Image count")
ax.set_title("MMOTU — images per class (8_layers/)")
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

minority = min(class_counts, key=class_counts.get)
majority = max(class_counts, key=class_counts.get)
ratio    = class_counts[majority] / class_counts[minority]
print(f"Minority class: {minority} ({class_counts[minority]})")
print(f"Majority class: {majority} ({class_counts[majority]})")
print(f"Imbalance ratio: {ratio:.1f}x  — consider class-weighted loss or oversampling")

In [ ]:
# --- Image size statistics (sample 200 images for speed) ---
random.seed(0)
sample_paths = random.sample(all_images, min(200, len(all_images)))

widths, heights = [], []
for p in sample_paths:
    w, h = Image.open(p).size
    widths.append(w)
    heights.append(h)

print(f"Sampled {len(sample_paths)} images")
print(f"Width   — min={min(widths):4d}  max={max(widths):4d}  mean={sum(widths)/len(widths):6.1f}")
print(f"Height  — min={min(heights):4d}  max={max(heights):4d}  mean={sum(heights)/len(heights):6.1f}")

unique_sizes = sorted(set(zip(widths, heights)))
print(f"\nUnique (W, H) pairs found: {len(unique_sizes)}")
for sz in unique_sizes[:10]:
    print(f"  {sz}")
if len(unique_sizes) > 10:
    print(f"  ... and {len(unique_sizes) - 10} more")

print("\nAll images will be resized to 256×256 during preprocessing.")

## Observations

*(Fill in after running the cells above.)*

- **Class imbalance:** ...
- **Image sizes:** ...
- **Mask format:** pixel values are `0` (background) or `>0` (lesion); binary threshold is `mask > 0`.
- **Preprocessing plan:** resize to 256×256, apply CLAHE to boost contrast, binarize masks.
- **Split strategy:** patient-disjoint 70/15/15 train/val/test (grouped by case prefix).